In [ ]:
import xarray as xr
import os
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.colors import Normalize, BoundaryNorm, ListedColormap, TwoSlopeNorm
import matplotlib.colors as mcolors
import matplotlib.animation as animation
from matplotlib.patches import Circle
import matplotlib.dates as mdates
from matplotlib import colors
from matplotlib.cm import get_cmap
import matplotlib.cm as cm
import numpy as np
import pandas as pd
from metpy.units import units
import metpy.calc as mpcalc
from metpy.constants import g
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
import cartopy.mpl.ticker as cticker
import dask
import xskillscore as xs
from scipy.fft import fft2, ifft2, fftfreq
from scipy.stats import pearsonr
import math
import glob
import re
import pickle
from scipy.ndimage import binary_dilation
from scipy.ndimage import gaussian_filter1d
from scipy.ndimage import gaussian_filter
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec

import warnings
warnings.filterwarnings('ignore')
import matplotlib as mpl
size = 14
mpl.rcParams.update({
    "font.size": size,
    "axes.titlesize": size,
    "axes.labelsize": size,
    "xtick.labelsize": size,
    "ytick.labelsize": size,
    "legend.fontsize": size,
    "legend.title_fontsize": size,
    "figure.titlesize": size
})


In [ ]:
Rd = 287.0 # J/kg/K 
cp = 1004.0 # J/kg/K 
p0 = 100000.0 # Pa 
theta_target = 320.0 # K
R = 6371.0  

In [ ]:
def load_era5_dataset_series_dask(
    start: str,
    end: str,
    base_path: str,
    file_prefix: str = '',
    freq: str = '3H'
):
    """
    Lazily load and concatenate ERA5 NetCDF datasets using Dask.

    Parameters:
        start (str): Start datetime string (e.g., '2020-05-01 00:00')
        end (str): End datetime string (e.g., '2020-05-03 23:00')
        base_path (str or Path): Base directory containing ERA5 files
        file_prefix (str): Optional prefix for filenames (e.g., 'Z' for 'Z20200503_16')
        freq (str): Time frequency of files (default = '3H')

    Returns:
        xr.Dataset: Concatenated Dask-backed xarray dataset along time dimension
    """

    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end)

    # Frequency is now optional
    datetimes = pd.date_range(start=start_dt, end=end_dt, freq=freq)

    file_paths = []
    for dt in datetimes:
        filename = f"{file_prefix}{dt.strftime('%Y%m%d_%H')}"
        file_path = Path(base_path) / f"{dt.year:04d}" / f"{dt.month:02d}" / filename

        if file_path.exists():
            file_paths.append(str(file_path))
        else:
            print(f"  Warning: Missing file {file_path}")

    if not file_paths:
        raise FileNotFoundError("No matching files found in the given range.")

    ds = xr.open_mfdataset(
        file_paths,
        combine='by_coords',
        chunks={}
    )

    return ds

## ACC related

In [ ]:
def compute_acc_rmse(ds_forecast, ds_analysis, ERA5_30D_mean_climatology):
    """
    Compute the anomaly correlation coefficient (ACC) between two datasets.
    
    Parameters:
        ds_forecast (xarray.DataArray): DataArray for IFS data
        ds_analysis (xarray.DataArray): DataArray for ERA data
    
    Returns:
        xarray.DataArray: Anomaly correlation coefficient over time
    """
    ds_forecast = ds_forecast.sel(lat=slice(35,75)).sel(lon=slice(-13, 40))
    ds_analysis = ds_analysis.sel(lat=slice(35,75)).sel(lon=slice(-13, 40))
    ERA5_30D_mean_climatology = ERA5_30D_mean_climatology.sel(lat=slice(35,75)).sel(lon=slice(-13, 40))

    times_era = ERA5_30D_mean_climatology.time.values
    times_fc  = ds_forecast.time.values
    times_an  = ds_analysis.time.values

    common_times = np.intersect1d(np.intersect1d(times_era, times_fc), times_an)
    # Step 3: Subset each dataset to those common times
    ERA5_30D_mean_climatology_subset = ERA5_30D_mean_climatology.sel(time=common_times)
    ds_forecast_subset = ds_forecast.sel(time=common_times)
    ds_analysis_subset = ds_analysis.sel(time=common_times)

    f_anom = (ds_forecast_subset - ERA5_30D_mean_climatology_subset)
    a_anom = (ds_analysis_subset - ERA5_30D_mean_climatology_subset)

    f_anom_avg = f_anom.mean(dim=['lat', 'lon'])
    a_anom_avg = a_anom.mean(dim=['lat', 'lon'])

    numerator = ((f_anom - f_anom_avg) * (a_anom - a_anom_avg)).sum(dim=['lat', 'lon'])

    denominator = (
        (((f_anom - f_anom_avg))**2).sum(dim=['lat', 'lon']) *
        (((a_anom - a_anom_avg))**2).sum(dim=['lat', 'lon']))**0.5

    acc = numerator / denominator

    #add the computation of RMSE
    rmse_time = xs.rmse(ds_forecast_subset.chunk({'time': -1}), ds_analysis_subset.chunk({'time': -1}), dim=['lat', 'lon'], skipna=True) 
    
    return acc, rmse_time

In [ ]:
def compute_acc_members(path_pattern, z500_era, clim, return_rmse=False):
    """
    Compute ACC at day 6 for all ensemble members matching path_pattern.
    Optionally return RMSE at day 6 as well.
    """
    results_acc = []
    results_rmse = []  # only used if return_rmse=True
    names = []

    for file in sorted(glob.glob(path_pattern)):
        ds = xr.open_dataset(file)
        z500 = ds.geopot.sel(plev=50000).sel(
            lat=slice(20, 80), lon=slice(-100, 45)
        ) / 9.81

        acc, rmse = compute_acc_rmse(z500, z500_era, clim)

        # ACC at day 6
        acc_val = float(acc.isel(time=6)) if "time" in acc.dims else float(acc[6])
        results_acc.append(acc_val)

        # RMSE at day 6 (if requested)
        if return_rmse:
            rmse_val = (
                float(rmse.isel(time=6)) if "time" in rmse.dims else float(rmse[6])
            )
            results_rmse.append(rmse_val)

        names.append(file.split("/")[-1])
        ds.close()

    if return_rmse:
        return names, results_acc, results_rmse
    else:
        return names, results_acc


## Wave Amplitude Error

In [ ]:
def invert_poisson_fft(zeta, dx, dy):
    ny, nx = zeta.shape
    kx = fftfreq(nx, dx.mean().magnitude) * 2 * np.pi
    ky = fftfreq(ny, dy.mean().magnitude) * 2 * np.pi
    kx, ky = np.meshgrid(kx, ky)

    k2 = kx**2 + ky**2
    k2[0, 0] = 1  # avoid divide by zero at zero wavenumber

    psi_hat = -fft2(zeta) / k2
    psi_hat[0, 0] = 0  # set mean to zero

    psi = np.real(ifft2(psi_hat))
    return psi

In [ ]:
def compute_wave_error(time, ds_1, ds_2):
    u_era = ds_1.U.sel(time=time, method='nearest').sel(lat=slice(10,80)).sel(lon=slice(-120, 20)).sel(plev=25000)
    v_era = ds_1.V.sel(time=time, method='nearest').sel(lat=slice(10,80)).sel(lon=slice(-120, 20)).sel(plev=25000)

    u_icon = ds_2.u.sel(time=time, method='nearest').sel(lat=slice(10,80)).sel(lon=slice(-120, 20)).sel(plev=25000)
    v_icon = ds_2.v.sel(time=time, method='nearest').sel(lat=slice(10,80)).sel(lon=slice(-120, 20)).sel(plev=25000)
    u_e = u_icon - u_era
    v_e = v_icon - v_era
    u_e.attrs['units'] = 'm/s'
    v_e.attrs['units'] = 'm/s'
    dx, dy = mpcalc.lat_lon_grid_deltas(u_e['lon'], u_e['lat'])

    zeta_e = mpcalc.vorticity(u_e, v_e, dx=dx, dy=dy, x_dim='lon', y_dim='lat')

    psi_e = invert_poisson_fft(zeta_e, dx, dy)
    dpsidx = np.gradient(psi_e, axis=1) / dx.mean().magnitude
    dpsidy = np.gradient(psi_e, axis=0) / dy.mean().magnitude

    laplacian_psi = (
        np.gradient(np.gradient(psi_e, axis=1), axis=1) / dx.mean().magnitude**2 +
        np.gradient(np.gradient(psi_e, axis=0), axis=0) / dy.mean().magnitude**2
    )
    E = 0.5 * (dpsidx**2 + dpsidy**2 - psi_e * laplacian_psi)

    E_xr = xr.DataArray(
        data=E,
        coords={"lat": u_era.lat, "lon": u_era.lon},
        dims=["lat", "lon"],
        name="Error Amplitude"
    )
    return E_xr

In [ ]:
def compute_ensemble_wae_std(times, ds_era_wind, ds_ensemble):
    """
    Computes standard deviation of WAE across all ensembles for given times.

    Returns:
        dict: {time_string: DataArray_of_std}
    """
    results = {}

    for t in times:
        # Compute WAE for each ensemble member
        wae_list = []
        for e in ds_ensemble.ensemble.values:
            ds_member = ds_ensemble.sel(ensemble=e)
            E = compute_wave_error(t, ds_era_wind, ds_member)  # your wave-error fn
            wae_list.append(E)

        # Stack along a new "ensemble" dimension
        wae_stack = xr.concat(wae_list, dim="ensemble")

        # Compute STD
        wae_std = wae_stack.std(dim="ensemble")

        results[t] = wae_std

    return results

## MCS

In [ ]:
def compute_MCS_intensity_pv(ds_icon, threshold_pv, lat_min, lat_max, lon_min, lon_max):
    """
    Compute total area (km²) where PV anomaly < threshold
    at plev = 250 hPa inside a fixed lat/lon box.
    """

    # ------------------------------------------
    # Extract PV anomaly field
    # ------------------------------------------
    pv_map = ds_icon #- mean_PV_250 #NO ANOMALIIIIEEEES !!!!!

    # ------------------------------------------
    # Grid properties
    # ------------------------------------------
    R = 6371e3  # Earth radius (m)

    lats = pv_map["lat"].values
    lons = pv_map["lon"].values

    lat_rad = np.radians(lats)
    lon_rad = np.radians(lons)

    dlat = np.diff(lat_rad).mean()
    dlon = np.diff(lon_rad).mean()

    lat_edges = lat_rad - dlat/2
    lat_edges = np.append(lat_edges, lat_edges[-1] + dlat)

    # Area per lat–lon grid cell
    cell_areas = R**2 * dlon * (np.sin(lat_edges[1:]) - np.sin(lat_edges[:-1]))
    cell_areas = cell_areas.reshape(-1, 1) * np.ones((1, len(lons)))

    # ------------------------------------------
    # Masks
    # ------------------------------------------
    mask_pv = (pv_map.values < threshold_pv)

    geo_mask = (
        (lats[:, None] >= lat_min) &
        (lats[:, None] <= lat_max) &
        (lons[None, :] >= lon_min) &
        (lons[None, :] <= lon_max)
    )

    mask = mask_pv & geo_mask

    # ------------------------------------------
    # Area calculation
    # ------------------------------------------
    area_m2 = float((mask * cell_areas).sum())
    area_km2 = area_m2 / 1e6

    return area_km2

## Warm conveyor belt

In [ ]:
def compute_WCB_ensemble_ascent(resolution_label, pmin=500, pmax=800, data_path='./output/ensemble/'):
    """
    Loop through all ensemble members, compute WCB ascent/outflow areas,
    and return a list of dicts with time + area data for later plotting.
    """
    file_pattern = os.path.join(data_path, f'trajectories_{resolution_label}_ens*.nc')
    file_list = sorted(glob.glob(file_pattern))
    
    all_results = []  # store (time, area_ascent, area_outflow, ens_name)
    
    for f in file_list:
        ens_name = os.path.basename(f).split('_')[-1].replace('.nc', '')
        print(f'Processing {ens_name}...')

        ds = xr.open_dataset(f)
        ds_WCB = select_WCB_trajectories_pressure_end_point(ds)

        mask_ascent = compute_WCB_mask(ds_WCB, pmin=pmin, pmax=pmax)

        area_ascent = compute_mask_area(mask_ascent)

        all_results.append({
            'ens': ens_name,
            'time': mask_ascent.time_abs.values,
            'area_ascent': area_ascent
        })
    
    return all_results

In [ ]:
def compute_WCB_mask(ds, pmin, pmax, lat_res=0.5, lon_res=0.5, radius_km=100):
    """
    Compute WCB ascent mask (500-800 hPa by default) and inflate parcels by radius_km.
    
    Parameters:
        ds: xarray.Dataset with trajectories
        lat_res, lon_res: grid resolution in degrees
        pmin, pmax: pressure bounds for ascent layer
        radius_km: inflation radius in km
    
    Returns:
        mask_da: xarray.DataArray of boolean mask (time_abs, lat, lon)
    """
    # Select WCB ascent layer
    mask_pressure = (ds["pressure"] <= pmax) & (ds["pressure"] >= pmin)
    ds_ascent = ds.where(mask_pressure, drop=True)

    # Define grid
    lat_grid = np.arange(-90, 90 + lat_res, lat_res)
    lon_grid = np.arange(-180, 180 + lon_res, lon_res)

    # Grid spacing in km
    dy = 111.0
    dx = 111.0 * np.cos(np.deg2rad(lat_grid.mean()))

    # Circle kernel for dilation
    radius_y = int(np.ceil(radius_km / dy / lat_res))
    radius_x = int(np.ceil(radius_km / dx / lon_res))
    yy, xx = np.ogrid[-radius_y:radius_y+1, -radius_x:radius_x+1]
    circle = (yy*dy*lat_res)**2 + (xx*dx*lon_res)**2 <= radius_km**2

    mask_list = []

    # Group by time_abs
    for t, subset in ds_ascent.groupby("time_abs"):
        lats = subset["lat"].values
        lons = subset["lon"].values

        mask = np.zeros((len(lat_grid), len(lon_grid)), dtype=bool)

        # Map parcels to nearest grid cells
        iy = np.searchsorted(lat_grid, lats) - 1
        ix = np.searchsorted(lon_grid, lons) - 1

        # Keep only valid indices
        valid = (iy >= 0) & (iy < len(lat_grid)) & (ix >= 0) & (ix < len(lon_grid))
        iy = iy[valid]
        ix = ix[valid]

        mask[iy, ix] = True

        # Inflate parcels
        mask = binary_dilation(mask, structure=circle)
        mask_list.append(mask)

    mask_da = xr.DataArray(
        np.stack(mask_list, axis=0),
        coords={"time_abs": list(ds_ascent.groupby("time_abs").groups.keys()),
                "lat": lat_grid,
                "lon": lon_grid},
        dims=("time_abs", "lat", "lon"),
        name="WCB_ascent_mask"
    )

    return mask_da


In [ ]:
def select_WCB_trajectories_pressure_end_point(ds):
    """
    Select WCB trajectories based on:
      1️⃣ Total pressure drop (≥ 600 hPa)
      2️⃣ Outflow region (lat/lon bounds)
      3️⃣ Final pressure below 400 hPa
    """

    # 1️⃣ Select trajectories with at least 600 hPa pressure decrease
    p_max = ds['pressure'].max(dim='time')
    p_min = ds['pressure'].min(dim='time')
    p_diff = p_max - p_min

    valid_trajectories = p_diff >= 600
    ds_WCB = ds.sel(trajectory=valid_trajectories)

    # 2️⃣ Remove duplicate trajectories, keeping the last occurrence
    traj_values = ds_WCB.trajectory.values
    _, index = np.unique(traj_values[::-1], return_index=True) # reverse array 
    index = len(traj_values) - 1 - index # map back to original indices 
    index = np.sort(index) # keep in original order

    ds_WCB_unique = ds_WCB.isel(trajectory=index)

    # 3️⃣ Select depending on lat/lon *and* final pressure of the outflow region
    end_idx = ds_WCB_unique.dims['time'] - 1
    end_lat = ds_WCB_unique['lat'].isel(time=end_idx)
    end_lon = ds_WCB_unique['lon'].isel(time=end_idx)
    end_p = ds_WCB_unique['pressure'].isel(time=end_idx)

    # Apply all conditions
    mask = (
        #(end_lat >= 20) & (end_lat <= 90) &     # latitude range
        #(end_lon >= -100) & (end_lon <= 40) &    # longitude range
        (end_p < 400)                           # final pressure condition
    )

    selected_trajectories = ds_WCB_unique['trajectory'].values[mask.values]

    # 4️⃣ Return only the selected WCB trajectories
    ds_WCB_selected = ds_WCB_unique.sel(trajectory=selected_trajectories)
    return ds_WCB_selected


In [1]:
def select_WCB_trajectories_pressure_end_June(ds):
    """
    Select WCB trajectories based on:
      1️⃣ Total pressure drop (≥ 600 hPa)
      2️⃣ Outflow region (lat/lon bounds)
      3️⃣ Final pressure below 400 hPa
    """

    # 1️⃣ Select trajectories with at least 600 hPa pressure decrease
    p_max = ds['pressure'].max(dim='time')
    p_min = ds['pressure'].min(dim='time')
    p_diff = p_max - p_min

    valid_trajectories = p_diff >= 600
    ds_WCB = ds.sel(trajectory=valid_trajectories)

    # 2️⃣ Remove duplicate trajectories, keeping the last occurrence
    traj_values = ds_WCB.trajectory.values
    _, index = np.unique(traj_values[::-1], return_index=True) # reverse array 
    index = len(traj_values) - 1 - index # map back to original indices 
    index = np.sort(index) # keep in original order

    ds_WCB_unique = ds_WCB.isel(trajectory=index)

    # 3️⃣ Select depending on lat/lon *and* final pressure of the outflow region
    end_idx = ds_WCB_unique.dims['time'] - 1
    end_lat = ds_WCB_unique['lat'].isel(time=end_idx)
    end_lon = ds_WCB_unique['lon'].isel(time=end_idx)
    end_p = ds_WCB_unique['pressure'].isel(time=end_idx)

    # Apply all conditions
    '''
    mask = (
        (end_lat >= 10) & (end_lat <= 90) &     # latitude range
        (end_lon >= -130) & (end_lon <= 40) &    # longitude range
        (end_p < 400)                           # final pressure condition
    )
    '''
    mask = (
    ((end_lat >= 50) |      # northward outflow
        (end_lon >= -40)      # eastward outflow
    )
    & (end_p < 400)
    & (end_lon >= -70))


    selected_trajectories = ds_WCB_unique['trajectory'].values[mask.values]

    # 4️⃣ Return only the selected WCB trajectories
    ds_WCB_selected = ds_WCB_unique.sel(trajectory=selected_trajectories)
    return ds_WCB_selected


In [ ]:
def compute_mask_area(mask_da):
    """
    Compute the total area covered by a boolean mask over time [million km²].
    """

    lat_vals = mask_da.lat.values
    lon_vals = mask_da.lon.values

    # Ensure latitudes are increasing
    if lat_vals[1] < lat_vals[0]:
        lat_vals = lat_vals[::-1]
        mask_da = mask_da.sel(lat=mask_da.lat[::-1])

    # Compute grid edges
    dlat = np.deg2rad(lat_vals[1] - lat_vals[0])
    dlon = np.deg2rad(lon_vals[1] - lon_vals[0])

    lat_edges = np.deg2rad(np.linspace(lat_vals[0] - 0.5 * np.rad2deg(dlat),
                                       lat_vals[-1] + 0.5 * np.rad2deg(dlat),
                                       len(lat_vals) + 1))

    # Compute cell areas (vectorized, independent of lon)
    cell_area = (R**2 * dlon *
                 (np.sin(lat_edges[1:, None]) - np.sin(lat_edges[:-1, None])))

    # Multiply by mask and sum
    area_over_time = []
    for t in mask_da.time_abs.values:
        mask_t = mask_da.sel(time_abs=t).values
        total_area = np.sum(cell_area * mask_t)
        area_over_time.append(total_area / 1e6)  # million km²

    return np.array(area_over_time)


In [ ]:
def compute_p_rate_99_for_resolution(resolution, base_path):
    """
    Compute 99th percentile ascent rate for all ensembles of one resolution.

    Returns:
        dict: {filename: p_rate_99}
    """
    file_pattern = (
        f"{base_path}/trajectories_{resolution}_ens*_remapcon.nc"
    )
    ensemble_files = sorted(glob.glob(file_pattern))

    results = {}

    for fn in ensemble_files:
        print(fn)
        ds = xr.open_dataset(fn)
        ds_WCB = select_WCB_trajectories_pressure_end_point(ds)

        pressure = ds_WCB["pressure"]#.values  # (traj, time)

        # --- time differences in hours ---
        time = ds_WCB["time"].values
        time_diff = np.diff(time) #/ np.timedelta64(1, "h")

        # --- pressure differences ---
        dp = np.diff(pressure, axis=1)

        # --- ascent rate (hPa/h) ---
        p_rate = -dp / time_diff

        # --- flatten + clean ---
        p_rate_all = p_rate.ravel()
        p_rate_all = p_rate_all[np.isfinite(p_rate_all)]

        # --- percentile ---
        if len(p_rate_all) == 0:
            p_rate_99_val = np.nan
        else:
            p_rate_99_val = np.quantile(p_rate_all, 0.95)

        results[fn] = p_rate_99_val
        ds.close()

    return results

In [ ]:
def compute_p_rate_99_for_resolution_5(resolution, base_path):
    """
    Compute 99th percentile ascent rate for all ensembles of one resolution.

    Returns:
        dict: {filename: p_rate_99}
    """
    file_pattern = (
        f"{base_path}/trajectories_{resolution}_ens*_remapcon.nc"
    )
    ensemble_files = sorted(glob.glob(file_pattern))

    results = {}

    for fn in ensemble_files:
        print(fn)
        ds = xr.open_dataset(fn)
        ds_WCB = select_WCB_trajectories_pressure_end_point(ds)

        pressure = ds_WCB["pressure"]#.values  # (traj, time)

        # --- time differences in hours ---
        time = ds_WCB["time"].values
        time_diff = np.diff(time) #/ np.timedelta64(1, "h")

        # --- pressure differences ---
        dp = np.diff(pressure, axis=1)

        # --- ascent rate (hPa/h) ---
        p_rate = -dp / time_diff

        # --- flatten + clean ---
        p_rate_all = p_rate.ravel()
        p_rate_all = p_rate_all[np.isfinite(p_rate_all)]

        # --- percentile ---
        if len(p_rate_all) == 0:
            p_rate_99_val = np.nan
        else:
            p_rate_99_val = np.quantile(p_rate_all, 0.97)

        results[fn] = p_rate_99_val
        ds.close()

    return results

In [1]:
def compute_p_rate_99_for_resolution_2(resolution, base_path):
    """
    Compute 99th percentile ascent rate for all ensembles of one resolution.

    Returns:
        dict: {filename: p_rate_99}
    """
    file_pattern = (
        f"{base_path}/trajectories_{resolution}_ens*_remapcon.nc"
    )
    ensemble_files = sorted(glob.glob(file_pattern))

    results = {}

    for fn in ensemble_files:
        print(fn)
        ds = xr.open_dataset(fn)
        ds_WCB = select_WCB_trajectories_pressure_end_point(ds)

        pressure = ds_WCB["pressure"]#.values  # (traj, time)

        # --- time differences in hours ---
        time = ds_WCB["time"].values
        time_diff = np.diff(time) #/ np.timedelta64(1, "h")

        # --- pressure differences ---
        dp = np.diff(pressure, axis=1)

        # --- ascent rate (hPa/h) ---
        p_rate = -dp / time_diff

        # --- flatten + clean ---
        p_rate_all = p_rate.ravel()
        p_rate_all = p_rate_all[np.isfinite(p_rate_all)]

        # --- percentile ---
        if len(p_rate_all) == 0:
            p_rate_99_val = np.nan
        else:
            p_rate_99_val = np.quantile(p_rate_all, 0.975)

        results[fn] = p_rate_99_val
        ds.close()

    return results

## Other

In [ ]:
def compute_irrotational_wind(U_in, V_in, min_speed=6):
    """
    Compute divergent (irrotational) wind using windspharm.
    U_in, V_in must still be on the full global grid.
    """

    # Ensure full globe is used
    from windspharm.xarray import VectorWind
    vw = VectorWind(U_in, V_in)

    # Global components
    uchi_full, vchi_full = vw.irrotationalcomponent()

    # Slice AFTER decomposition
    uchi = uchi_full.sel(lat=slice(lat_max_MCS, lat_min_MCS),
                         lon=slice(lon_min_MCS, lon_max_MCS))

    vchi = vchi_full.sel(lat=slice(lat_max_MCS, lat_min_MCS),
                         lon=slice(lon_min_MCS, lon_max_MCS))

    # Mask weak wind
    mag = np.sqrt(uchi**2 + vchi**2)
    mask = mag > min_speed

    return uchi.where(mask), vchi.where(mask)

In [ ]:
def track_cyclone_fast(ds, initial_time, initial_lat, initial_lon, search_radius_deg=1):

    pres = ds["pres_msl"].values
    totp = ds["tot_prec"].values
    conp = ds["rain_con"].values

    lats = ds.lat.values
    lons = ds.lon.values
    times = ds.time.values

    # --- Find start time index ---
    t0 = np.where(times == np.datetime64(initial_time))[0][0]

    # --- Initial indices ---
    ilat = np.abs(lats - initial_lat).argmin()
    ilon = np.abs(lons - initial_lon).argmin()

    dlat = np.abs(lats[1] - lats[0])
    dlon = np.abs(lons[1] - lons[0])

    rlat = int(search_radius_deg / dlat)
    rlon = int(search_radius_deg / dlon)

    nt = len(times) - t0

    # --- Preallocate output arrays ---
    out_time = np.empty(nt, dtype='datetime64[ns]')
    out_lat  = np.empty(nt)
    out_lon  = np.empty(nt)
    out_slp  = np.empty(nt)
    out_totp = np.empty(nt)
    out_conp = np.empty(nt)

    for i, it in enumerate(range(t0, len(times))):

        lat_min = max(0, ilat - rlat)
        lat_max = min(len(lats), ilat + rlat + 1)
        lon_min = max(0, ilon - rlon)
        lon_max = min(len(lons), ilon + rlon + 1)

        # --- Minimum SLP ---
        window_slp = pres[it, lat_min:lat_max, lon_min:lon_max]
        flat_idx = np.nanargmin(window_slp)
        dlat_i, dlon_i = np.unravel_index(flat_idx, window_slp.shape)

        ilat = lat_min + dlat_i
        ilon = lon_min + dlon_i

        # --- Precipitation means ---
        out_totp[i] = np.nanmean(
            totp[it, lat_min:lat_max, lon_min:lon_max]
        )
        out_conp[i] = np.nanmean(
            conp[it, lat_min:lat_max, lon_min:lon_max]
        )

        out_time[i] = times[it]
        out_lat[i]  = lats[ilat]
        out_lon[i]  = lons[ilon]
        out_slp[i]  = pres[it, ilat, ilon]

    return pd.DataFrame({
        "time": out_time,
        "lat": out_lat,
        "lon": out_lon,
        "pres_msl": out_slp,
        "tot_prec_sum": out_totp,
        "rain_con_sum": out_conp
    })


In [ ]:
def track_cyclone_era(ds, initial_time, initial_lat, initial_lon, search_radius_deg=1):
    """
    Fast cyclone tracker using NumPy indexing.
    Also computes mean(LSP + CP) within the tracking window.
    """

    pres = ds["pres_msl"]
    lsp  = ds["LSP"]
    cp   = ds["CP"]

    lats = pres.lat.values
    lons = pres.lon.values
    times = pres.time.values

    slp = pres.values
    lsp_val = lsp.values
    cp_val  = cp.values

    # Find start time index
    initial_time = np.datetime64(initial_time)
    t0 = np.where(times == initial_time)[0][0]

    # Nearest grid point to initial guess
    ilat = np.abs(lats - initial_lat).argmin()
    ilon = np.abs(lons - initial_lon).argmin()

    # Grid spacing (assumes regular grid)
    dlat = np.abs(lats[1] - lats[0])
    dlon = np.abs(lons[1] - lons[0])

    rlat = int(search_radius_deg / dlat)
    rlon = int(search_radius_deg / dlon)

    results = []

    for it in range(t0, len(times)):

        lat_min = max(0, ilat - rlat)
        lat_max = min(len(lats), ilat + rlat + 1)
        lon_min = max(0, ilon - rlon)
        lon_max = min(len(lons), ilon + rlon + 1)

        # --- SLP minimum for tracking ---
        window_slp = slp[it, lat_min:lat_max, lon_min:lon_max]
        flat_idx = np.nanargmin(window_slp)
        dlat_i, dlon_i = np.unravel_index(flat_idx, window_slp.shape)

        ilat = lat_min + dlat_i
        ilon = lon_min + dlon_i

        # --- Mean total precipitation in same window ---
        window_prec = (
            lsp_val[it, lat_min:lat_max, lon_min:lon_max] +
            cp_val[it,  lat_min:lat_max, lon_min:lon_max]
        )
        tot_prec_mean = np.nanmean(window_prec)

        results.append({
            "time": pd.to_datetime(times[it]),
            "lat": float(lats[ilat]),
            "lon": float(lons[ilon]),
            "MSL": float(slp[it, ilat, ilon]),
            "tot_prec_sum": float(tot_prec_mean)
        })

    return pd.DataFrame(results)


In [2]:
def track_cyclone_ifs(ds, initial_time, initial_lat, initial_lon, search_radius_deg=1):
    """
    Fast cyclone tracker using NumPy indexing.
    Also computes mean(LSP + CP) within the tracking window.
    """

    pres = ds["msl"]

    lats = pres.lat.values
    lons = pres.lon.values
    times = pres.time.values

    slp = pres.values

    # Find start time index
    initial_time = np.datetime64(initial_time)
    t0 = np.where(times == initial_time)[0][0]

    # Nearest grid point to initial guess
    ilat = np.abs(lats - initial_lat).argmin()
    ilon = np.abs(lons - initial_lon).argmin()

    # Grid spacing (assumes regular grid)
    dlat = np.abs(lats[1] - lats[0])
    dlon = np.abs(lons[1] - lons[0])

    rlat = int(search_radius_deg / dlat)
    rlon = int(search_radius_deg / dlon)

    results = []

    for it in range(t0, len(times)):

        lat_min = max(0, ilat - rlat)
        lat_max = min(len(lats), ilat + rlat + 1)
        lon_min = max(0, ilon - rlon)
        lon_max = min(len(lons), ilon + rlon + 1)

        # --- SLP minimum for tracking ---
        window_slp = slp[it, lat_min:lat_max, lon_min:lon_max]
        flat_idx = np.nanargmin(window_slp)
        dlat_i, dlon_i = np.unravel_index(flat_idx, window_slp.shape)

        ilat = lat_min + dlat_i
        ilon = lon_min + dlon_i

        results.append({
            "time": pd.to_datetime(times[it]),
            "lat": float(lats[ilat]),
            "lon": float(lons[ilon]),
            "msl": float(slp[it, ilat, ilon]),
        })

    return pd.DataFrame(results)
